# Vision Transformer

**0 前言**<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.1 为什么图像任务也可以使用Transformer？<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.2 从CNN到ViT：视觉建模思路的变化<br>

**1 ViT模型的核心思想与整体结构**<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.1 ViT到底想解决什么问题？<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.2 ViT整体结构与数据流<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.3 ViT和原始Transformer的关系<br>

**2 ViT模型的输入：从图像到Patch序列**<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.1 图像为什么要切成Patch？<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.2 Patch Flatten与Linear Projection<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.3 图像输入在ViT中的张量形状变化<br>

**3 分类标记CLS Token与分类输出**<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.1 为什么要额外加入一个分类标记？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.2 CLS Token如何汇总整张图像的信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.3 分类头MLP Head如何输出类别概率？<br>

**4 ViT中的位置编码**<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.1 图像Patch为什么也需要位置信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.2 ViT中的可学习位置编码<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.3 位置编码与图像二维结构的关系<br>

<font color="red">**5 ViT中的Transformer Encoder主干**</font><br>
&nbsp;&nbsp;&nbsp;&nbsp;5.1 ViT中的多头自注意力机制<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.2 ViT中的前馈神经网络MLP Block<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.3 残差连接与Layer Normalization<br>

**6 ViT的训练、特点与局限**<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.1 ViT为什么通常需要较大数据量？<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.2 ViT和CNN的归纳偏置差异<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.3 ViT的优势、局限与常见变体<br>


## 5 ViT中的Transformer Encoder主干

前面几章我们已经把ViT的输入部分基本讲清楚了：

- 图像会被切成patch
- 每个patch会被映射成向量
- 序列最前面会加入CLS Token
- 每个位置还会加入位置编码

到这一步为止，模型终于拿到了一串既包含内容信息、也包含位置信息的输入token。

但这还只是开始。真正让ViT具备图像理解能力的，不是“输入被组织好了”这件事本身，而是：**这些token进入Transformer Encoder之后，会怎样彼此交换信息、逐层更新表示、最后形成更强的全局语义。**

这一章就专门来拆ViT里的Transformer Encoder主干。我们会先看最核心的多头自注意力机制，再看它后面的前馈网络，最后再把残差连接和LayerNorm放回完整的数据流里理解。


### 5.1 ViT中的多头自注意力机制

学到这里，我们终于进入了ViT最核心的计算部分。

前面我们已经知道，一张图像经过切patch、线性映射、加入CLS Token和位置编码之后，会变成一串输入token。但这时这些token还只是“各自带着局部信息的初始表示”。如果模型想真正理解整张图像，它就不能只让每个patch孤立地保留自己的局部内容，而必须让不同patch之间建立联系。

这时要解决的问题就是：

**一个patch怎样知道，图像里还有哪些其他patch和自己有关？它又应该从哪些位置吸收更多信息？**

ViT里负责解决这个问题的核心机制，就是**多头自注意力（Multi-Head Self-Attention, MHSA）**。

不过，这里先不要一上来就记公式。先抓住它最本质的作用：

**自注意力做的事情，就是让序列中的每个token都能去“看”整条序列里的其他token，并按相关性从它们那里收集信息。**

放在ViT里，这句话可以直接翻译成：

**每个patch都可以参考整张图中其他patch的表示，从而把原本局部的视觉信息逐步变成带有全局上下文的信息。**

#### 5.1.1 为什么ViT特别需要这种全局交互？

我们先从一个直觉问题出发。

假设图像里有一只猫。切成patch之后，可能会出现这样的情况：

- 有的patch只包含猫耳朵的一部分
- 有的patch只包含猫眼睛
- 有的patch只包含猫身体的一部分纹理
- 还有很多patch可能只是背景

如果这些patch彼此之间完全不交流，那么某一个只看到“局部毛发纹理”的patch，其实很难单独知道自己属于“猫”的哪个部分，也很难知道整个图里到底出现了什么目标。

但如果这个patch可以同时参考其他patch，比如猫耳朵、猫眼睛、猫轮廓所在的patch，那么它就能在更大的上下文中理解自己。也就是说，**ViT不是只让patch保留自己的局部观察，而是希望它们彼此交换信息，逐步形成全局理解。**

这正是自注意力最重要的价值。

#### 5.1.2 什么叫“自注意力”？为什么叫self？

这里的“self”，指的是：

**序列中的每个token，都是从同一条输入序列中的其他token那里取信息。**

也就是说，在ViT的Encoder里，输入是一串：

$$
[CLS, patch_1, patch_2, \ldots, patch_N]
$$

那么不管是`patch_7`还是`CLS Token`，它们在做注意力计算时，参考的对象都来自这同一串token本身，而不是来自外部另一条序列。

这和Decoder里的交叉注意力不同。交叉注意力是“去看另一条序列”；而这里的自注意力是“在自己这条序列内部彼此看”。

#### 5.1.3 在ViT里，一个patch具体是怎样从其他patch那里收集信息的？

现在我们把问题说得再具体一点。

假设当前输入序列记为：

$$
X \in \mathbb{R}^{(N+1) \times d_{model}}
$$

这里：

- $N$ 是patch数量
- 额外的 $1$ 对应CLS Token
- $d_{model}$ 是每个token表示的维度

多头自注意力不会直接拿这个输入去做“相关性比较”，而是先通过三个不同的线性变换，把它映射成：

$$
Q = XW^Q, \quad K = XW^K, \quad V = XW^V
$$

其中：

- $Q$ 是 Query，也就是查询向量
- $K$ 是 Key，也就是键向量
- $V$ 是 Value，也就是值向量

这里尤其要注意，**Q、K、V都不是某一个token单独拥有的三个固定属性，而是整条输入序列分别经过三组线性层之后得到的三组表示。** 也就是说，每个token都会有自己对应的query、key、value向量。

如果只看第 $i$ 个token，那么它会有：

- 一个查询向量 $q_i$
- 一个键向量 $k_i$
- 一个值向量 $v_i$

直观地说，可以这样理解：

- Query：我现在想从别人那里找什么信息
- Key：我这里提供的信息大致是什么类型
- Value：如果别人决定关注我，真正要取走的内容是什么

这不是严格数学定义，但对初学者建立直觉非常有帮助。

#### 5.1.4 注意力分数到底在算什么？

当第 $i$ 个token想更新自己时，它会拿自己的 $q_i$ 去和整条序列里所有token的 $k_j$ 做相似度比较，得到一组分数。这些分数反映的是：

**对当前这个token来说，序列里的哪些位置更值得关注。**

在矩阵形式下，这一步通常写成：

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

这里先不要被整条公式吓住，我们先解释它在干什么。

第一步：

$$
QK^T
$$

会计算出“每个token对所有token的相关性分数”。

第二步：

$$
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)
$$

会把这些分数归一化成注意力权重，也就是“应该分别关注多少”。

第三步：

$$
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

会用这些权重对整条序列里的Value向量做加权求和，得到每个token更新后的表示。

这里有一个你前面已经问过、也是最值得澄清的点：**这里乘的 $V$ 不是某一个词或者某一个patch单独的embedding，而是整条输入序列对应的Value矩阵。**

也就是说，自注意力本质上不是“看一下别人”，而是：

**根据相关性，从整条输入序列中重新收集信息，生成每个位置的新表示。**

#### 5.1.5 用一个patch例子理解“重新建模”

假设某个patch主要看到了猫的一只耳朵。单独看这个patch，它的信息是局部且不完整的。

但在注意力计算中，它可能对下面这些patch分配较高权重：

- 包含另一只耳朵的patch
- 包含眼睛的patch
- 包含猫脸轮廓的patch

而对纯背景patch分配较低权重。

这样一来，这个“耳朵patch”更新后的表示，就不再只是“耳朵局部纹理”，而会变成一个融合了整张图中相关区域信息的新表示。换句话说，它已经开始带有“这是猫的一部分”这种更全局的上下文语义。

这就是为什么我们常说：**注意力会对输入表示进行上下文化（contextualization）或重新建模。**

#### 5.1.6 为什么叫多头注意力？

到现在为止，我们讲的其实还是“单头”的直觉：一个token去看其他token，并收集信息。

但真实的Transformer不会只做一次这样的注意力，而是会并行地做很多组，也就是多个头。每个头都会有自己的一套：

- $W^Q$
- $W^K$
- $W^V$

因此，不同的头可以学习关注不同类型的关系。例如在ViT里：

- 有的头可能更关注局部纹理相似性
- 有的头可能更关注远距离区域之间的语义对应
- 有的头可能更偏向帮助CLS Token汇总全局信息

这也是“多头”最重要的意义：**不是只用一种方式看整张图，而是允许模型从多个不同子空间、不同关系角度同时观察输入。**

如果每个头的输出维度记为 $d_{head}$，头的数量记为 $h$，那么通常有：

$$
d_{model} = h \times d_{head}
$$

各个头分别算完之后，会先拼接起来，再通过一个输出线性层映射回模型维度。

这里要特别注意两点：

- 单个头内部处理的是较小的子维度，不是整个 `d_model`
- 但整个多头注意力模块对外的输入和输出形状通常仍然都是 `(序列长度, d_model)`

这也是为什么它可以方便地和残差连接配合。

#### 5.1.7 多头自注意力的输入输出形状到底是什么？

这一点最好现在就说清楚，因为初学者特别容易在维度这里糊涂。

假设输入张量形状是：

$$
(B, N+1, d_{model})
$$

这里：

- $B$ 是batch大小
- $N+1$ 是序列长度，也就是CLS Token加上所有patch
- $d_{model}$ 是每个token的特征维度

那么多头自注意力模块的输出形状通常仍然是：

$$
(B, N+1, d_{model})
$$

这并不意味着模块内部什么都没变，而是说明：

- 每个token的表示内容被更新了
- 但序列长度没变
- 模型维度也通常保持不变

所以更准确地说，**注意力改变的是表示内容，而不是把序列长度变长或变短。**

#### 5.1.8 CLS Token在这里也参与同样的注意力计算

还有一个特别值得再强调一次的点：`CLS Token`并不是在注意力之外单独处理的。它和所有patch token一样，也会生成自己的Query、Key、Value，并参与整条序列的自注意力计算。

这意味着：

- patch可以关注CLS
- CLS也可以关注patch
- 它们会在同一个注意力机制里彼此交换信息

这也是为什么CLS Token最终能够汇总全局信息，因为它不是最后临时收集信息，而是在每一层自注意力里都在不断与其他patch交互。

#### 5.1.9 这一节真正应该记住什么

到这里，ViT中的多头自注意力机制可以压缩成下面几句话：

- 它的核心作用是让每个patch都能从整条序列中的其他patch那里收集信息
- 注意力权重决定了每个位置应该更关注哪些位置
- 这种信息收集不是取某一个patch，而是对整条序列的Value做加权汇总
- 多头机制让模型能从多个不同角度并行建模patch之间的关系
- 输出形状通常与输入形状相同，但每个token的表示内容已经被上下文化了

理解了这一点之后，下一步自然就是继续问：**注意力把不同patch之间的信息混合起来之后，后面为什么还要接一个前馈神经网络？** 这就是下一节 `5.2 ViT中的前馈神经网络MLP Block` 要解决的问题。


**Nano Banana绘图提示词**

画一张教学图，主题是“ViT中的多头自注意力机制如何让Patch彼此交换信息”。整体采用从左到右布局，白底，中文标签完整，适合深度学习课件。左侧画输入序列 `[CLS, patch_1, patch_2, ..., patch_N]`，标注形状 `(B, N+1, d_model)`；中间画一个重点放大的当前patch，例如 `patch_i`，从它发出多条注意力连线指向其他patch和CLS Token，线条粗细不同，用来表示注意力权重大小；旁边标注“根据相关性分配注意力权重”；再往右画出“加权求和所有Value向量”，标注 `softmax(QK^T / sqrt(d_k)) V`；最右侧画出更新后的 `patch_i` 表示，并说明“融合全局上下文后的新表示”。在图的下半部分再画“多头并行”示意，用 3 到 4 个并行小头表示不同头关注不同关系，最后拼接并线性映射回 `d_model`。要求不同颜色区分 Query、Key、Value 和不同注意力头，强调“每个patch都能看整条序列”“输入输出形状通常相同”，整体风格简洁、工整、教学感强。


### 5.2 ViT中的前馈神经网络MLP Block

上一节我们已经看到，多头自注意力机制最核心的作用，是让每个token都能够参考整条序列里的其他token，从而把原本局部的patch表示变成带有全局上下文的信息表示。

但如果你继续追问一步，就会出现一个很自然的问题：

**既然注意力已经让不同patch之间完成了信息交互，为什么Transformer Encoder里后面还要再接一个前馈神经网络，也就是MLP Block？**

这个问题非常关键。因为如果这里不讲清楚，MLP Block就很容易被误解成“顺手加上的几层全连接”，看起来可有可无。实际上，它在Encoder里承担的是一个和自注意力不同、但同样重要的角色。

先用一句话概括它的作用：

**自注意力负责让不同token彼此交换信息，而MLP Block负责对每个token自己的表示做进一步的非线性加工。**

也就是说，这两个模块解决的问题并不一样。

#### 5.2.1 为什么信息交互之后还不够？

我们先看一个直觉。

经过自注意力之后，每个patch token都已经从其他patch那里收集了一些信息。例如，一个原本只看到猫耳朵的patch，现在可能已经融合了猫眼睛、猫轮廓、背景等其他区域的信息。

但注意一点：**这一步主要做的是从哪里取信息、取多少信息的问题。**

换句话说，自注意力更擅长做的是：

- 判断哪些token和当前token更相关
- 按注意力权重把其他token的信息加权汇总进来

可是一旦这些信息已经被收集进来，模型还需要继续做另一件事：

**把这个已经融合了上下文的表示，再进一步变换成更适合当前任务的特征。**

这就像你先把各处搜集来的材料放到桌上，但还需要再整理、归纳、提炼，才能真正形成一个更有用的中间结果。MLP Block做的，正是这种对单个token表示本身的进一步加工。

#### 5.2.2 MLP Block处理的是token之间的关系，还是token自身的表示？

这里最好分得非常清楚。

- 自注意力：主要负责token和token之间的交互
- MLP Block：主要负责每个token自身表示的变换

也就是说，MLP Block并不会像自注意力那样，再让一个patch去看另一个patch。它通常是**对序列中的每个token分别做同样的前馈变换**。

这句话非常重要，因为很多初学者会误以为MLP Block又是在做一轮patch之间的信息混合。其实不是。更准确地说，在进入MLP Block时，每个token已经带着上一阶段混合好的上下文表示了；而MLP Block的任务，是在这个基础上对每个token做更强的特征提炼。

#### 5.2.3 MLP Block的结构通常长什么样？

在ViT里，MLP Block通常可以理解为一个两层前馈网络，中间配合非线性激活函数。最常见的一种写法是：

$$
\text{MLP}(x) = W_2 \, \sigma(W_1 x + b_1) + b_2
$$

其中：

- $x$ 是某个token当前的输入表示
- $W_1, b_1$ 是第一层线性变换的参数
- $\sigma$ 是非线性激活函数，例如GELU
- $W_2, b_2$ 是第二层线性变换的参数

从结构上看，它通常是这样的流程：

1. 先把维度从 `d_model` 扩大到更高维
2. 经过非线性激活
3. 再把维度映射回 `d_model`

所以很多资料里也会把它画成：

`Linear -> GELU -> Linear`

这里有时还会加 Dropout，但从理解主线来说，最核心的骨架就是这三步。

#### 5.2.4 为什么中间要先升维，再降回去？

这又是一个很值得解释的问题。如果最终输入输出维度都还是 `d_model`，那为什么不直接用一层线性变换，而要先把维度拉高再降回来？

原因在于：**更高的中间维度可以给模型提供更强的表示能力。**

你可以把它理解成：先把当前token的特征投影到一个更宽的特征空间里，在那里做更丰富的组合和变换，再压回模型需要的维度。这样模型就不只是做一个简单、受限的线性调整，而是能表达更复杂的非线性关系。

在很多ViT实现里，这个中间维度常常会取成：

$$
d_{ff} = 4 \times d_{model}
$$

当然，这不是必须死记的唯一比例，但它是很常见的设置。

#### 5.2.5 为什么一定要有非线性激活？

如果只有线性层，没有激活函数，那么连续堆叠多个线性变换，本质上仍然可以合并成一个更大的线性变换。这样模型的表达能力会受到很大限制。

所以中间加入激活函数，意义在于：

**让模型能够学习更复杂的非线性特征变换，而不是只做简单的线性重排。**

在ViT和很多Transformer实现里，常见的激活函数不是ReLU，而是GELU。你现在不一定要深究GELU的精确数学形式，但最好知道：它的作用是给MLP Block引入非线性，使得每个token的表示可以被更灵活地加工。

#### 5.2.6 用一个直观类比理解：自注意力像交流，MLP像消化

如果前面的解释还是有点抽象，可以用一个很直观的类比来理解。

你可以把一个Encoder层里的这两个核心部分看成：

- 自注意力：每个token去和别人交流，收集外部信息
- MLP Block：每个token把收集到的信息在自己内部进一步消化、整理、重编码

这样你就能看出，为什么这两部分缺一不可。

如果只有MLP，没有自注意力，那么每个patch只能各自处理自己，无法获得全局上下文。

如果只有自注意力，没有MLP，那么虽然token之间可以交换信息，但每个token对这些信息的内部变换和提炼能力又会不足。

所以，一个负责交流，一个负责加工，两者组合起来，才构成了Transformer Encoder层中最核心的工作链条。

#### 5.2.7 在张量形状上，MLP Block会怎么变化？

这一点也最好讲清楚。

假设经过注意力之后，输入到MLP Block的张量形状是：

$$
(B, N+1, d_{model})
$$

那么第一层线性变换通常会把最后一维从 `d_model` 扩大到 `d_ff`，得到：

$$
(B, N+1, d_{ff})
$$

然后经过激活函数，再通过第二层线性层映射回：

$$
(B, N+1, d_{model})
$$

这里要特别注意两点：

- MLP Block不会改变序列长度 `N+1`
- 它只在每个token的特征维度上做变换

也就是说，它不改变有多少个token，而是改变每个token内部怎样表示。

#### 5.2.8 CLS Token也会经过同样的MLP Block吗？

会。

这一点和自注意力阶段一样，CLS Token并不是特殊跳过MLP Block的。更准确地说：

- patch token会经过同样的前馈网络
- CLS Token也会经过同样的前馈网络

区别不在于谁有没有经过MLP，而在于它们携带的信息内容不同。

所以，CLS Token不仅会在注意力中汇总全局信息，也会在MLP Block里进一步把这种全局信息重编码成更适合分类任务的表示。

#### 5.2.9 这一节真正应该记住什么

到这里，ViT中的MLP Block可以压缩成下面几句话：

- 自注意力负责token之间的信息交互
- MLP Block负责对每个token自身的表示做进一步非线性加工
- 它通常由两层线性层和中间的激活函数组成
- 常见做法是先升维到 `d_ff`，再降回 `d_model`
- 它不会改变序列长度，只会改变每个token内部的特征表示

理解了这一点之后，下一步自然就是把注意力和MLP放回一个完整的Encoder层里，看它们前后怎样通过残差连接和LayerNorm组织起来。这就是下一节 `5.3 残差连接与Layer Normalization` 要解决的问题。


**Nano Banana绘图提示词**

画一张教学图，主题是“ViT中的MLP Block如何对每个Token做非线性加工”。整体采用从左到右布局，白底，中文标签完整，适合深度学习课件。左侧画输入token序列，标注形状 `(B, N+1, d_model)`，并特别说明“每个token已经融合上下文信息”；中间画单个token经过MLP Block的结构，依次标注“线性层1：d_model -> d_ff”“GELU激活”“线性层2：d_ff -> d_model”；在图中强调“对每个token分别应用同样的MLP，不进行token之间的信息混合”；右侧画输出序列，标注形状仍为 `(B, N+1, d_model)`，并说明“序列长度不变，token表示被进一步非线性提炼”。下方加一个对比框，写明“自注意力：token之间交流”“MLP Block：token内部加工”。要求用不同颜色区分输入、升维、中间激活、降维输出，整体风格简洁、工整、教学感强。


### 5.3 残差连接与Layer Normalization

前两节我们已经分别拆开看了Encoder层里的两个核心计算模块：

- 多头自注意力负责让token之间交换信息
- MLP Block负责对每个token自己的表示做进一步非线性加工

但如果你现在把视角拉回到“完整的一个Encoder层”，马上就会遇到两个新的问题：

**这些模块前后到底是怎样连起来的？为什么每个模块旁边还要配一个残差连接和Layer Normalization？**

这两个部件看起来不像注意力那样显眼，也不像MLP那样直观，但它们其实对Transformer能否稳定训练、能否有效传递信息非常关键。

先用一句话概括：

- **残差连接**帮助模型在保留原始信息的同时学习增量变化
- **Layer Normalization**帮助模型稳定数值分布，使训练更平稳

这两个部件不是装饰，而是Transformer主干能够深层堆叠的重要基础。

#### 5.3.1 先看整体：一个Encoder层到底长什么样？

在ViT里，一个标准的Transformer Encoder层，通常可以概括成下面这条主线：

1. 输入先经过LayerNorm
2. 再进入多头自注意力
3. 然后和原输入做残差相加
4. 再经过一次LayerNorm
5. 进入MLP Block
6. 再和前一步结果做残差相加

如果用更紧凑的形式写，就是：

$$
X' = X + \text{MHSA}(\text{LN}(X))
$$

$$
Y = X' + \text{MLP}(\text{LN}(X'))
$$

这里你会看到，残差连接和LayerNorm并不是只出现一次，而是分别围绕着“注意力子层”和“MLP子层”各出现一遍。

这就是ViT中非常常见的Pre-LN结构，也就是“先归一化，再进入子层”。

#### 5.3.2 为什么需要残差连接？

我们先讲残差连接，因为它的直觉比较好建立。

假设某个token经过多头自注意力之后，得到了一份更新后的表示。如果没有残差连接，那么这个token的新表示就只能完全依赖注意力模块的输出。这样做有一个潜在问题：

**模型在每一层都会被迫把旧表示彻底替换成新表示，原来的信息不容易直接保留下来。**

而残差连接的做法是：

$$
输出 = 原输入 + 子层输出
$$

这意味着，模型不是非得推翻原来的表示，而是可以在原表示的基础上，学习“需要补充什么、调整什么”。

换句话说，残差连接让子层更像是在学习一个**增量修正**，而不是每次都重新发明一套全新表示。

从直觉上，你可以把它理解成：

- 原输入提供一条稳定的信息主干
- 注意力或MLP只负责在这条主干上增加新的修正项

这样模型在层数变深时，信息就不容易被层层冲淡或意外破坏。

#### 5.3.3 残差连接除了保留信息，还有什么好处？

除了“保留原始信息”这个直观好处之外，残差连接还有一个非常重要的工程意义：

**它能让深层网络更容易训练。**

因为当网络层数变多时，如果每一层都必须学会一个完整而复杂的映射，优化会变得困难；而如果每一层只需要学一个相对于输入的小修正，训练往往就会稳定得多。

这也是为什么残差结构不仅出现在Transformer里，在ResNet等深层视觉网络里也非常关键。

所以你可以把残差连接理解成两层含义：

- 表示层面：帮助保留已有信息
- 优化层面：帮助深层网络更容易学习

#### 5.3.4 什么是Layer Normalization？

接下来再看LayerNorm。

在深层网络中，一个很常见的问题是：随着层数加深，不同层输出的数值分布可能会越来越不稳定。有的维度值可能特别大，有的可能特别小，这会让后续训练变得困难。

Layer Normalization做的事情，可以先用一句尽量直白的话来理解：

**它会对每个token自己的特征维度做归一化，让这个token的表示分布保持在一个更稳定的范围里。**

这里尤其要注意：**LayerNorm不是在token之间做归一化，而是在单个token内部、沿着特征维度做归一化。**

如果一个token的表示是：

$$
x \in \mathbb{R}^{d_{model}}
$$

那么LayerNorm会基于这个向量内部各维度的统计量，对它做归一化，然后再乘上可学习的缩放参数、加上可学习的平移参数。

你现在不一定需要死记它的完整公式，但最好抓住它的功能：

- 让不同层输入到子模块之前的数值尺度更稳定
- 减轻训练过程中的数值波动
- 帮助模型更平稳地优化

#### 5.3.5 为什么这里不用Batch Normalization，而要用LayerNorm？

这是一个初学者很容易问到的问题。

Batch Normalization通常依赖一个batch里不同样本的统计量，而Transformer处理的是序列结构，不同batch大小、不同序列情况都可能影响这种统计方式。

LayerNorm则不依赖整个batch的统计，而是只针对单个token自己的特征维度进行归一化，因此它更适合Transformer这类序列模型，也更适合在训练和推理时保持一致的行为。

所以，在Transformer和ViT里，更常见的是LayerNorm，而不是BatchNorm。

#### 5.3.6 为什么是“LN -> 子层 -> 残差相加”这样的顺序？

这一点也很值得说明。

在ViT中常见的是Pre-LN结构，也就是先做LayerNorm，再进入注意力或MLP子层。这样做的一个重要好处是：

**子层每次接收到的输入分布会更稳定。**

也就是说，在进入MHSA之前，先把输入归一化一下；在进入MLP之前，也先把当前表示归一化一下。这样注意力模块和前馈网络不需要每一层都去适应非常剧烈的数值波动，训练通常会更稳定。

然后子层输出再通过残差和原输入相加，使模型能够在保留主干信息的同时加入新变化。

所以，这个顺序不是随便摆的，而是兼顾了：

- 子层输入的稳定性
- 深层训练的可优化性
- 原始信息的保留

#### 5.3.7 用一个直观类比来理解这两个部件

如果你觉得这些结构有点工程化，可以用一个直观类比来帮助理解。

可以把一个Encoder层想象成这样：

- 自注意力和MLP像两个加工模块
- LayerNorm像进入加工站前先把材料整理到比较统一、稳定的状态
- 残差连接像保留一条原始材料的主干通道，让新加工结果是在原有基础上叠加，而不是完全替换

这样你就能看出，LayerNorm更偏向“稳定输入”，残差连接更偏向“保留主干并学习修正”。两者虽然不起眼，但对整个结构的可训练性都非常重要。

#### 5.3.8 在张量形状上，这些操作会改变维度吗？

一般不会改变整体形状。

假设某一层的输入张量形状是：

$$
(B, N+1, d_{model})
$$

那么：

- LayerNorm之后，形状仍然是 `(B, N+1, d_model)`
- MHSA之后，形状仍然是 `(B, N+1, d_model)`
- 残差相加之后，形状仍然是 `(B, N+1, d_model)`
- MLP之后，最后又回到 `(B, N+1, d_model)`

这也是为什么残差相加能直接成立，因为相加两边的张量形状必须一致。

所以更准确地说，这些模块主要改变的是**表示内容**和**数值分布**，而不是外部张量形状。

#### 5.3.9 这一节真正应该记住什么

到这里，ViT中Encoder层里的残差连接和LayerNorm可以压缩成下面几句话：

- 残差连接让模型在保留原始信息的同时学习增量修正
- 它不仅有助于信息保留，也有助于深层网络训练
- LayerNorm会在单个token内部沿特征维度做归一化
- 它帮助子层输入保持更稳定的数值分布
- ViT中常见的是Pre-LN结构：先LayerNorm，再进入子层，再做残差相加

到这里，ViT里的一个完整Transformer Encoder层就已经讲清楚了：输入先带着内容和位置进入Encoder，然后在每一层中先通过自注意力做全局信息交互，再通过MLP做局部非线性加工，并依靠残差连接和LayerNorm稳定地层层向前传播。


**Nano Banana绘图提示词**

画一张教学图，主题是“ViT中一个完整Transformer Encoder层的结构：MHSA、MLP、残差连接与LayerNorm”。整体采用从上到下的数据流布局，白底，中文标签完整，适合深度学习课件。最上方是输入序列，标注形状 `(B, N+1, d_model)`；接着画第一段子层：`LayerNorm -> 多头自注意力 MHSA -> 残差相加`，用一条旁路箭头从输入直接绕到残差加号处，清楚表示残差连接；然后画第二段子层：`LayerNorm -> MLP Block -> 残差相加`，同样用旁路箭头表示第二个残差连接；最下方是输出序列，标注形状仍为 `(B, N+1, d_model)`。在图右侧加一个简短说明框：`残差连接：保留主干信息，学习增量修正`，`LayerNorm：稳定每个token的特征分布`。要求不同颜色区分主路径和残差旁路，整体风格简洁、工整、教学感强。
